# LASSER study: multi-split evaluation, controls, ablations and analysis

This notebook runs the full experiment set for the paper and analyses it.

| Tier | Arms | Splits | Runs |
|---|---|---|---|
| Core | `GEARS`, `LASSER` | 1–5 | 10 |
| Controls | `GEARS-kNN8` (density-matched Pearson graph), `LASSER-random` (learned edges rewired at random, same in-degrees and weights) | 1–5 | 10 |
| Ablations | `LASSER-noS`, `LASSER-Eonly`, `LASSER-addonly`, `LASSER-embinit`, `LASSER-m2`, `LASSER-m10` | 1–3 | 18 |

**Kaggle settings:** Accelerator **GPU T4 x2**, **Internet on**.

**How to run.** Each run takes about 85 min. With two GPUs one Kaggle session (12 h) finishes about 14 runs, so the study needs **about 3 sessions**.

1. Use *Save Version → Save & Run All (Commit)*, so it runs in the background for up to 12 h. Workers stop taking new jobs before the time limit, and the analysis cells then run on whatever has finished.
2. For the next session, add the previous version's **output** as an input (*Add Input → Your Work → this notebook*) and set `RESUME_FROM` in cell 3 to its `study` folder. Finished jobs are skipped automatically.
3. Repeat until the status table shows every job done. The analysis cells (section B) can also be run alone, CPU only, on the final output.

## A. Setup and experiments

In [ ]:
# 1. Dependencies (GEARS is vendored inside lasser/; matplotlib, scipy, pandas are preinstalled on Kaggle).
!pip install -q torch_geometric scanpy pytest

In [ ]:
# 2. Code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# 3. Settings. Edit RESUME_FROM in sessions 2, 3, ... (the previous version's output 'study' folder).
import os, shutil, time
T0 = time.time()                                   # session start, for the time budget
SESSION_HOURS = 11.5                               # Kaggle limit is 12 h; leave time for analysis + saving
ANALYSIS_RESERVE_HOURS = 0.5
STUDY_DIR = "/kaggle/working/study"                # saved as notebook output (runs, cache, tables, figures)
DATA_DIR = "/tmp/lasser_data"                      # NOT saved (several GB); re-downloaded each session
RESUME_FROM = None                                 # e.g. "/kaggle/input/lasser-study/study"
RUN_FULL_TESTS = False                             # True: also the dataset tests (~20 min)

os.makedirs(STUDY_DIR, exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, STUDY_DIR, dirs_exist_ok=True)
    shutil.rmtree(os.path.join(STUDY_DIR, "claims"), ignore_errors=True)   # claims are per session
    print("resumed from", RESUME_FROM)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 4. Fast tests (seconds; no data): GEARS-printout parser, random-edge control, job table, analysis, figures.
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_05_parsing.py {REPO_DIR}/tests/test_06_study.py -q -p no:cacheprovider --tb=short
if RUN_FULL_TESTS:
    %env LASSER_DATA_DIR={DATA_DIR}
    %env LASSER_TEST_DIR=/tmp/lasser_tests
    !cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -q -p no:cacheprovider --tb=short 2>&1 | tail -n 30

In [ ]:
# 5. The experiment table (priority order: core, controls, ablations) and what is already done.
import pandas as pd
from lasser.study import ARM_DESCRIPTIONS, DEFAULT_PLAN, make_jobs, study_status, expected_minutes

BASE = dict(dataset="norman", split="simulation", epochs=20, device="cuda",
            data_dir=DATA_DIR, out_dir=f"{STUDY_DIR}/runs", cache_dir=f"{STUDY_DIR}/cache", repo_dir=REPO_DIR)
PLAN = DEFAULT_PLAN                                 # list of (arm, splits); edit to add or drop runs
jobs = make_jobs(PLAN, BASE)
display(pd.DataFrame([(a, ", ".join(map(str, s)), ARM_DESCRIPTIONS[a]) for a, s in PLAN],
                     columns=["arm", "splits", "what it tests"]))
status = study_status(jobs)
print(f"{int(status.done.sum())}/{len(status)} runs done; ~{expected_minutes(BASE['out_dir']):.0f} min per run")
status.pivot_table(index="arm", columns="seed", values="done", aggfunc="first").reindex([a for a, _ in PLAN])

In [ ]:
# 6. Download data and build every split and GEARS graph once (one subprocess, ~10 min the first time).
from lasser.study import prepare_data
if not study_status(jobs).done.all():
    prepare_data(jobs)

In [ ]:
# 7. Run the remaining jobs: one worker process per GPU, progress printed every 10 min.
#    Workers stop taking jobs when the next one would pass the session budget.
from lasser.study import launch
budget = SESSION_HOURS - ANALYSIS_RESERVE_HOURS - (time.time() - T0) / 3600
if not study_status(jobs).done.all():
    status = launch(jobs, budget_hours=budget, poll_minutes=10)
status = study_status(jobs)
print(f"{int(status.done.sum())}/{len(status)} runs done")
fails = f"{BASE['out_dir']}/failures.csv"
if os.path.exists(fails):
    display(pd.read_csv(fails).tail(5))
status.pivot_table(index="arm", columns="seed", values="test_pearson_delta", aggfunc="first").reindex([a for a, _ in PLAN])

## B. Analysis

Everything below reads the saved run folders, so it works on partial results and can be rerun alone (CPU) on a
previous version's output: set `STUDY_DIR` to that folder and run from here.

Metrics are eval.py's paper metrics (Roohani et al. 2024): Pearson Δ over all genes and over the top-20 DE genes,
MSE on the top-20 DE genes, normalised MSE (ratio of means, robust to tiny denominators) and the fraction of top-20
DE genes predicted in the wrong direction. Statistics:
- **Across splits** (the unit of replication): mean ± sd, paired *t*-test, Wilcoxon signed-rank (with 5 splits its
  smallest possible two-sided *p* is 0.0625), Holm correction across the comparisons in a table.
- **Across perturbations**: paired (split, perturbation) differences, share improved, Wilcoxon, and a 95 % CI from a
  bootstrap that resamples splits and then perturbations within splits.

In [ ]:
# 8. Load results; reference baselines (no perturbation, additive) scored with the same eval.py functions.
import numpy as np
from lasser import analysis as A
from lasser import figures as F
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)

RESULTS_DIR, FIG_DIR = f"{STUDY_DIR}/results", f"{STUDY_DIR}/figures"
runs = A.load_runs(f"{STUDY_DIR}/runs")
long = A.metrics_long(runs)
H5AD = f"{DATA_DIR}/norman/perturb_processed.h5ad"
if os.path.exists(H5AD):
    long = pd.concat([long, A.paper_reference_baselines(runs, H5AD)], ignore_index=True)
else:
    print("no h5ad found; skipping the no-perturbation/additive baselines")
CORE = [a for a in ["GEARS", "LASSER", "GEARS-kNN8", "LASSER-random"] if a in set(runs.arm)]
REFS = [a for a in ["No-perturb", "Additive"] if a in set(long[long.family == "paper"].arm)]
ABL = [a for a in ["LASSER", "LASSER-noS", "LASSER-Eonly", "LASSER-addonly", "LASSER-embinit",
                   "LASSER-m2", "LASSER-m10"] if a in set(runs.arm)]
print(runs.groupby("arm").seed.apply(sorted))

In [ ]:
# 9. Table 1: headline metrics, mean ± sd over splits (overall and per test subgroup).
SCOPES = ["overall"] + A.SUBGROUPS
t1 = A.summary_table(long, A.HEADLINE, scopes=["overall"], arms=REFS + CORE)
display(t1)
t1_sub = A.summary_table(long, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"],
                         scopes=A.SUBGROUPS, arms=REFS + CORE)
display(t1_sub)

In [ ]:
# 10. Table 2: paired tests across splits. Improvement > 0 means the arm is better (sign handled per metric).
comparisons = [("GEARS", ["LASSER", "GEARS-kNN8", "LASSER-random"]),     # vs the published model
               ("GEARS-kNN8", ["LASSER"]),                              # learned vs density-matched Pearson
               ("LASSER-random", ["LASSER"])]                           # learned vs random edges, same degrees
t2 = pd.concat([A.paired_tests(long, ref, [a for a in arms if a in CORE], A.HEADLINE, scopes=SCOPES)
                for ref, arms in comparisons if ref in CORE], ignore_index=True)
t2["p_t_holm"] = A._holm(t2["p_t"].to_numpy())
show = ["arm", "ref", "scope", "metric", "n_splits", "ref_mean", "arm_mean", "improvement", "ci95_low",
        "ci95_high", "rel_improvement_pct", "splits_better", "p_t", "p_wilcoxon", "p_t_holm"]
display(t2[t2.scope == "overall"][show].round(4))
display(t2[(t2.scope != "overall") & (t2.metric == "pearson_delta")][show].round(4))

In [ ]:
# 11. Table 3: per-perturbation paired analysis (all splits pooled; bootstrap resamples splits).
pp = A.per_perturbation(runs)
t3 = pd.concat([A.paired_perturbation(pp, ref, arm, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"])
                for ref, arms in comparisons if ref in CORE for arm in arms if arm in CORE], ignore_index=True)
display(t3.round(4))

In [ ]:
# 12. Table 4: ablations vs LASSER, on the splits where both exist (1-3).
t4 = A.paired_tests(long, "LASSER", [a for a in ABL if a != "LASSER"], A.HEADLINE, scopes=["overall"])
display(t4[show].round(4) if len(t4) else "no ablation runs yet")

In [ ]:
# 13. Where does the gain come from? Output genes isolated vs connected in GEARS's graph A0.
gg = A.gene_group_metrics(runs)
t5 = A.gene_group_paired(gg, "GEARS", [a for a in CORE if a != "GEARS"])
display(t5.round(4))

In [ ]:
# 14. The learned graph: structure and label-free checks per split; stability of new edges across splits;
#     how many new edges plain correlation (density-matched kNN) would also have found.
gt = A.graph_table(runs)
display(gt.groupby("arm")[[c for c in ["num_edges", "isolated_genes", "degree_median", "degree_max", "h2_hit_added",
                                        "h2_hit_random", "h2_hit_a0", "link_auc", "link_auc_raw", "jaccard_to_a0"]
                           if c in gt]].agg(["mean", "std"]).round(3))
for arm in [a for a in ["LASSER", "LASSER-random"] if a in CORE]:
    jac, counts = A.edge_stability(runs, arm)
    off = jac.to_numpy(dtype=float)[~np.eye(len(jac), dtype=bool)]
    print(f"{arm}: mean Jaccard of new-edge sets between splits = {np.nanmean(off):.3f}; "
          f"pairs in >=3 splits: {int((counts.n_splits >= 3).sum()) if len(counts) else 0}")
display(A.overlap_with_knn8(runs))
consensus = A.consensus_edges(runs, "LASSER", min_splits=3)
display(consensus.head(30))

In [ ]:
# 15. Figures (PDF + PNG in study/figures).
paths = []
paths += F.fig_headline(long, REFS + CORE, FIG_DIR)
paths += F.fig_subgroups(long, CORE, FIG_DIR)
paths += F.fig_per_perturbation(pp, FIG_DIR)
paths += F.fig_forest(t2, FIG_DIR, "fig_forest_core", "Paired improvement over splits")
if len(t4):
    paths += F.fig_forest(t4, FIG_DIR, "fig_forest_ablations", "Ablations vs LASSER (splits 1-3)")
paths += F.fig_gene_groups(t5, FIG_DIR)
paths += F.fig_training(A.training_curves(runs), CORE, FIG_DIR)
paths += F.fig_graphs(gt, runs, FIG_DIR)
from IPython.display import Image, display as _d
for p in paths:
    if p.endswith(".png"):
        _d(Image(p, width=900))

In [ ]:
# 16. Export tables (CSV + LaTeX) and pack results/figures/run tables into one zip for download.
A.export_tables({"table1_headline": t1, "table1_subgroups": t1_sub, "table2_paired_splits": t2,
                 "table3_paired_perturbations": t3, "table4_ablations": t4, "table5_gene_groups": t5,
                 "graph_stats": gt, "consensus_edges": consensus}, RESULTS_DIR)
shutil.copy(f"{STUDY_DIR}/runs/runs.csv", RESULTS_DIR)
!cd {STUDY_DIR} && zip -qr /kaggle/working/lasser_study_results.zip results figures runs/runs.csv runs/jobs.json
print("results:", sorted(os.listdir(RESULTS_DIR)))
print("download: /kaggle/working/lasser_study_results.zip")

## C. Reporting checklist

- Report every arm as **mean ± sd over the 5 splits**, the paired improvement with its 95 % CI, and the number of
  splits that improved (Table 2). With 5 splits, give the paired *t*-test *p* and say Wilcoxon cannot go below
  0.0625. Use Holm-corrected *p* when claiming significance across several metrics or subgroups.
- The **controls** carry the claim. `LASSER > GEARS` alone may be "more edges help". `LASSER > GEARS-kNN8`
  (same density, plain correlation) and `LASSER > LASSER-random` (same degrees and weights, random partners) show
  that the *learned* edges matter.
- Show the reference baselines (No-perturb, Additive for combinations) next to the models, as the GEARS paper does.
- Report the per-perturbation view (Table 3, Fig. 3): the share of test perturbations improved and the
  split-resampling bootstrap CI.
- Graph analysis (Fig. 6, consensus edges): H2 hit rate of new edges vs degree-matched random, stability across
  splits, and the overlap with the kNN Pearson graph. Consensus edges are the starting point for the
  "reasoning about gene relationships" section.
- Known limitation to state: GEARS's co-expression GNN passes messages only for the first cell of each batch
  (upstream implementation, kept unchanged for a fair comparison).